In [64]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [65]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [66]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(x_dev)


In [67]:
train_df.shape[0]

276

# Defining Model

In [68]:
#model = 'XGBoost'
#model = 'RandomForest'
model = 'TabPFN'

# Fit Model

In [69]:
if model == 'TabPFN':

    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train_scale, y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev_scale)

elif model == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 5,
        'eta': 0.3,
        'eval_metric': 'rmse',
    }

    # Train the model
    num_boost_round = 5
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)

elif model == 'RandomForest':

    # Set the parameters for the Random Forest model
    params = {
            'n_estimators': 11,
            'max_depth': 5,
            'min_samples_split': 2,
            'min_samples_leaf': 4,
            'random_state': 42,
    }

    predictions = RandomForestRegressor(**params).fit(x_train_scale, y_train).predict(x_dev_scale)


In [70]:
predictions

array([3883.8103, 2181.7324, 2174.3381, 2229.0996, 3985.961 , 2243.853 ,
       3270.4026, 3300.8215, 3295.8608, 3312.6807, 2852.2012, 2855.6055,
       2854.8784, 2855.63  , 2907.641 , 2726.153 , 2744.1553, 2763.5925,
       2739.1401, 2740.8643, 2734.703 , 2724.2285, 2734.1829, 2716.3418,
       2731.2769, 2731.3748, 2743.463 , 2751.5051, 2749.3928, 2745.423 ,
       2744.1843, 2744.8945, 2744.2283, 2766.9258, 2766.442 , 2758.7222,
       2760.1355, 2760.11  , 2757.6655, 2764.6199, 2761.2449, 3133.9685,
       3108.5615, 3144.858 , 3134.4963, 3142.7358, 3140.6458, 3124.5432,
       3097.9106, 3088.3174, 3113.2258, 3098.559 , 3097.0896, 3108.5588,
       3114.554 , 3125.4985, 3071.9253, 3079.4614, 3077.0913, 3099.881 ,
       3090.6877, 2971.6038, 2972.646 , 2955.9568, 2908.4575, 2944.11  ,
       2961.665 , 2936.9192, 2941.4988, 2960.356 , 2944.8796, 2945.7756,
       2938.0073, 2874.7603, 2876.3506, 2885.9329, 2870.7378, 2868.9702,
       2867.7234, 2875.1006, 2862.769 , 3114.934 , 

In [71]:
print(y_dev)

0     4161.4
1     1836.4
2     2509.8
3     2867.4
4     5277.7
       ...  
88    2937.9
89    3028.8
90    2860.7
91    2816.5
92    2978.6
Name: PullTest (N), Length: 93, dtype: float64


# Check Validation Data

In [72]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=predictions,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model}) by Category - Data-Driven Training"
)

# Check Validation Loss and R2

In [73]:

# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(root_mean_squared_error(y_dev, predictions))**2
R2   = r2_score(y_dev, predictions)



print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  122.09
RMSE: 202.55
R2: 0.66


# Maybe Feature Importance?


In [74]:
#feature_names = x_train.columns.tolist()
# Fix: tell SHAP what the model's feature names are 
#regressor.feature_names_in_ = np.array(feature_names)

# Calculate SHAP values
#shap_values = interpretability.shap.get_shap_values(
#    estimator=regressor,
#    test_x=test_x,
#    attribute_names=feature_names,
#    algorithm="permutation",
#)

# Create visualization
#fig = interpretability.shap.plot_shap(shap_values)

In [75]:
#x_dev.columns


# Cross Validation

In [ ]:
cross_df = pd.read_csv("data/train_dev_data.csv")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mae_list = []
rmse_list = []
R2_list = []

for fold, (train_index, val_index) in enumerate(skf.split(cross_df["Sample ID"], cross_df["Category"])):
    x_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column) 
    x_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

    x_tr_scale = sc.fit_transform(X=x_tr)
    x_val_scale = sc.transform(x_val)

    if model == 'XGBoost':
        dtrain = xgb.DMatrix(x_tr_scale, label=y_tr)
        dval = xgb.DMatrix(x_val_scale, label=y_val)

        params = {
            'objective': 'reg:squarederror',
            'max_depth': 1,
            'eta': 0.57,
            'eval_metric': 'rmse'
        }

        num_boost_round = 20
        bst = xgb.train(params, dtrain, num_boost_round)

        preds = bst.predict(dval)

    elif model == 'RandomForest':

        params = {
            'n_estimators': 11,
            'max_depth': 5,
            'min_samples_split': 2,
            'min_samples_leaf': 4,
            'random_state': 42,
        }

        preds = RandomForestRegressor(**params).fit(x_tr_scale, y_tr).predict(x_val_scale)
        
    else:
        regressor = TabPFNRegressor()
        regressor.fit(x_tr, y_tr)

        preds = regressor.predict(x_val)

    mae  = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    R2   = r2_score(y_val, preds)

    mae_list.append(mae) 
    rmse_list.append(rmse) 
    R2_list.append(R2)

    # Categories
    categories= cross_df.iloc[val_index]["Category"].values

    plot_visualizer(
        true_vals=y_val,
        pred_vals=preds,
        categories=categories,
        title=f"Fold {fold+1}: True vs Prediction ({model}) by Category - Cross-Validation Data-Driven Training"
    )

    print(f"\nFold {fold+1}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", R2)

mae_mean = np.mean(mae_list) 
rmse_mean = np.mean(rmse_list) 
R2_mean = np.mean(R2_list) 



Fold 1
MAE : 121.69049105257602
RMSE: 186.81721187084392
R²  : 0.6340417995056008



Fold 2
MAE : 138.77436919341218
RMSE: 283.44597773148706
R²  : 0.7014452213348172



Fold 3
MAE : 127.33780352618241
RMSE: 209.66846719006563
R²  : 0.78451618691722



Fold 4
MAE : 111.85529521220441
RMSE: 149.54616327938228
R²  : 0.7122334393274238



Fold 5
MAE : 107.17089776862157
RMSE: 173.63941221550584
R²  : 0.6610089922659921


In [77]:
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")

mean MAE:  121.37
mean RMSE: 200.62
mean R²:   0.70


In [78]:
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")


mean MAE:  121.37
mean RMSE: 200.62
mean R²:   0.70


In [79]:
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")

mean MAE:  121.37
mean RMSE: 200.62
mean R²:   0.70


# Select between XGB and RandomForest

In [80]:
model_type = "xgbregressor"
#model_type = "random_forest" 

# Model Creation

In [81]:
def create_model(model_type, params):
    if model_type == "xgbregressor":
        return XGBRegressor(
            objective='reg:squarederror',
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            n_estimators=params['n_estimators'],
            eval_metric='rmse'
        )
    elif model_type == "random_forest":
        return RandomForestRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_samples_split=params['min_samples_split'],
            min_samples_leaf=params['min_samples_leaf'],
            random_state=42,
        )


# Parameter Selection

In [82]:
if model_type == "xgbregressor":
    param_grid = []
    max_depth_values = range(1, 3)
    eta_values = [0.5, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57]
    n_estimators_values = range(1, 26)

    for md in max_depth_values:
        for eta in eta_values:
            for ne in n_estimators_values:
                param_grid.append({
                    'max_depth': md,
                    'learning_rate': eta,
                    'n_estimators': ne
                })

elif model_type == "random_forest":
    param_grid = []
    n_estimators_values = range(9, 14)
    max_depth_values = range(4, 7)
    min_samples_split_values = [4,5,6,7,8,9]
    min_samples_leaf_values = range(3, 6)

    for ne in n_estimators_values:
        for md in max_depth_values:
            for mss in min_samples_split_values:
                for msl in min_samples_leaf_values:
                    param_grid.append({
                        'n_estimators': ne,
                        'max_depth': md,
                        'min_samples_split': mss,
                        'min_samples_leaf': msl
                    })


# Gridsearch

In [83]:
best_rmse = float("inf")
best_params = None
best_model = None

for params in param_grid:
    print(f"\nTesting params: {params}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    mae_list = []
    rmse_list = []
    R2_list = []

    for fold, (train_index, val_index) in enumerate(
            skf.split(cross_df["Sample ID"], cross_df["Category"])):

        X_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column)
        X_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

        X_tr_scale = sc.fit_transform(X_tr)
        X_val_scale = sc.transform(X_val)

        model = create_model(model_type, params)
        
        model.fit(X_tr_scale, y_tr)
        preds = model.predict(X_val_scale)

        mae = mean_absolute_error(y_val, preds) 
        rmse = root_mean_squared_error(y_val, preds) 
        R2 = r2_score(y_val, preds) 

        mae_list.append(mae) 
        rmse_list.append(rmse) 
        R2_list.append(R2)

    mae_mean = np.mean(mae_list) 
    rmse_mean = np.mean(rmse_list) 
    R2_mean = np.mean(R2_list) 

    if rmse_mean < best_rmse:
        best_rmse = rmse_mean
        best_params = params
        best_model = model

        print("\n==============================")
        print(" BEST MODEL FOUND ")
        print("==============================")
        print(f"Best RMSE:   {best_rmse:.2f}")
        print(f"Best Params: {best_params}")



Testing params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 1}

 BEST MODEL FOUND 
Best RMSE:   309.78
Best Params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 1}

Testing params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 2}

 BEST MODEL FOUND 
Best RMSE:   300.48
Best Params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 2}

Testing params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 3}

 BEST MODEL FOUND 
Best RMSE:   285.60
Best Params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 3}

Testing params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 4}

 BEST MODEL FOUND 
Best RMSE:   257.32
Best Params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 4}

Testing params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 5}

 BEST MODEL FOUND 
Best RMSE:   248.04
Best Params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators': 5}

Testing params: {'max_depth': 1, 'learning_rate': 0.5, 'n_estimators'

In [84]:
print("\n==============================")
print(" BEST MODEL FOUND ")
print("==============================")
print(f"Best RMSE:   {best_rmse:.2f}")
print(f"Best Params: {best_params}")

#==============================
# BEST MODEL FOUND XGBoost
#==============================
#Best RMSE:   245.17
#Best Params: {'max_depth': 5, 'learning_rate': 0.3, 'n_estimators': 5}

#==============================
# BEST MODEL FOUND RandomForest
#==============================
#Best RMSE:   208.64
#Best Params: {'n_estimators': 11, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 4}


 BEST MODEL FOUND 
Best RMSE:   204.38
Best Params: {'max_depth': 1, 'learning_rate': 0.57, 'n_estimators': 20}
